# TAAC 2026 UNI-REC — Fixed Stable PyTorch Baseline

This notebook fixes the instability you saw where training loss became negative and validation AUC collapsed to 0.5.

Main fixes:

1. Use `BCEWithLogitsLoss`, not `BCELoss`.
2. The model returns **raw logits**, not sigmoid probabilities.
3. Sigmoid is used only during evaluation/inference.
4. Labels are explicitly converted to binary `0/1` and checked.
5. Added gradient clipping, weight decay, dropout, and best-AUC checkpointing.

Run this notebook top-to-bottom in Google Colab.

In [9]:
# Install dependencies in Colab
!pip -q install datasets scikit-learn tqdm pandas matplotlib

In [10]:
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, log_loss
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


## 1. Load dataset and inspect labels

In [11]:
ds = load_dataset('TAAC2026/data_sample_1000')
print(ds)

split_name = list(ds.keys())[0]
data = ds[split_name]
print('Using split:', split_name)
print('Rows:', len(data))
print('Columns:', len(data.column_names))
print(data.column_names[:20])

DatasetDict({
    train: Dataset({
        features: ['user_id', 'item_id', 'label_type', 'label_time', 'timestamp', 'user_int_feats_1', 'user_int_feats_3', 'user_int_feats_4', 'user_int_feats_15', 'user_int_feats_48', 'user_int_feats_49', 'user_int_feats_50', 'user_int_feats_51', 'user_int_feats_52', 'user_int_feats_53', 'user_int_feats_54', 'user_int_feats_55', 'user_int_feats_56', 'user_int_feats_57', 'user_int_feats_58', 'user_int_feats_59', 'user_int_feats_60', 'user_int_feats_62', 'user_int_feats_63', 'user_int_feats_64', 'user_int_feats_65', 'user_int_feats_66', 'user_int_feats_80', 'user_int_feats_82', 'user_int_feats_86', 'user_int_feats_89', 'user_int_feats_90', 'user_int_feats_91', 'user_int_feats_92', 'user_int_feats_93', 'user_int_feats_94', 'user_int_feats_95', 'user_int_feats_96', 'user_int_feats_97', 'user_int_feats_98', 'user_int_feats_99', 'user_int_feats_100', 'user_int_feats_101', 'user_int_feats_102', 'user_int_feats_103', 'user_int_feats_104', 'user_int_feats_105'

In [13]:
# Inspect one row and label distribution
row0 = data[0]
print('First row preview:')

for k in list(row0.keys())[:25]:
    v = row0[k]
    print(k, type(v), v if not isinstance(v, list) else v[:5])

label_col = 'label_type'
labels_raw = [r[label_col] for r in data]

print('\nRaw label values:', sorted(set(labels_raw)))
print(pd.Series(labels_raw).value_counts().sort_index())

First row preview:
user_id <class 'int'> 3864676
item_id <class 'int'> 103989760
label_type <class 'int'> 1
label_time <class 'int'> 1772725413
timestamp <class 'int'> 1772725140
user_int_feats_1 <class 'int'> 4
user_int_feats_3 <class 'int'> 1753
user_int_feats_4 <class 'int'> 6
user_int_feats_15 <class 'list'> [928, 556, 538, 739, 94]
user_int_feats_48 <class 'int'> 42
user_int_feats_49 <class 'int'> 2
user_int_feats_50 <class 'int'> 1
user_int_feats_51 <class 'int'> 56
user_int_feats_52 <class 'int'> 24
user_int_feats_53 <class 'int'> 101
user_int_feats_54 <class 'int'> 810
user_int_feats_55 <class 'int'> 41
user_int_feats_56 <class 'int'> 681
user_int_feats_57 <class 'int'> 111
user_int_feats_58 <class 'int'> 1
user_int_feats_59 <class 'int'> 3
user_int_feats_60 <class 'list'> [2]
user_int_feats_62 <class 'list'> [6, 4]
user_int_feats_63 <class 'list'> [9, 15, 45, 36]
user_int_feats_64 <class 'list'> [2, 50, 23]

Raw label values: [1, 2]
1    876
2    124
Name: count, dtype: int64


### Important label handling

`BCEWithLogitsLoss` requires binary targets in `[0, 1]`. If `label_type` contains values other than 0/1, this notebook maps positive labels to 1 and non-positive labels to 0. This prevents negative BCE losses.

In [ ]:
def to_binary_label(v):
    # Safe conversion for CVR-style binary prediction.
    # If the dataset labels are already 0/1, this leaves them unchanged.
    return 1.0 if float(v) > 0 else 0.0

binary_labels = [to_binary_label(v) for v in labels_raw]
print('Binary label values:', sorted(set(binary_labels)))
print(pd.Series(binary_labels).value_counts().sort_index())
assert set(binary_labels).issubset({0.0, 1.0})

## 2. Identify feature groups

In [ ]:
all_cols = data.column_names
sample = data[0]

scalar_sparse_cols = []
list_sparse_cols = []
dense_cols = []
seq_cols = []
ignore_cols = {'label_type', 'label_time', 'timestamp'}

for col in all_cols:
    if col in ignore_cols:
        continue
    val = sample[col]
    if col.startswith('user_dense_feats'):
        dense_cols.append(col)
    elif col.startswith('domain_') and '_seq_' in col:
        seq_cols.append(col)
    elif col.startswith('user_int_feats') or col.startswith('item_int_feats') or col in ['user_id', 'item_id']:
        if isinstance(val, list):
            list_sparse_cols.append(col)
        else:
            scalar_sparse_cols.append(col)

print('Scalar sparse:', len(scalar_sparse_cols), scalar_sparse_cols[:15])
print('List sparse:', len(list_sparse_cols), list_sparse_cols[:15])
print('Dense:', len(dense_cols), dense_cols[:15])
print('Sequence:', len(seq_cols), seq_cols[:15])

## 3. Choose a stable baseline feature subset

For the midterm, we first validate the pipeline using scalar sparse features. Once this is stable, you can add list, dense, and sequence features.

In [ ]:
# Compact starter set. This keeps training fast and interpretable.
preferred_cols = [
    'user_id', 'item_id',
    'user_int_feats_1', 'user_int_feats_3', 'user_int_feats_4',
    'item_int_feats_5', 'item_int_feats_6', 'item_int_feats_7',
    'item_int_feats_8', 'item_int_feats_9', 'item_int_feats_10'
]
feature_cols = [c for c in preferred_cols if c in scalar_sparse_cols]

# Fallback: if any preferred cols are missing, use the first few scalar sparse columns.
if len(feature_cols) < 4:
    feature_cols = scalar_sparse_cols[:10]

print('Using feature columns:', feature_cols)

In [ ]:
# Split first, then build vocabularies using training data only to avoid validation leakage.
split = data.train_test_split(test_size=0.2, seed=SEED)
train_data = split['train']
val_data = split['test']
print('Train rows:', len(train_data), 'Val rows:', len(val_data))

def build_vocab(dataset, cols, min_count=1):
    vocabs = {}
    for col in cols:
        counts = {}
        for row in dataset:
            v = row[col]
            if v is None:
                continue
            v = int(v)
            counts[v] = counts.get(v, 0) + 1
        kept = sorted([v for v, c in counts.items() if c >= min_count])
        # 0 = unknown/OOV/missing
        vocabs[col] = {v: i + 1 for i, v in enumerate(kept)}
    return vocabs

vocabs = build_vocab(train_data, feature_cols)
for col in feature_cols:
    print(f'{col}: vocab size including OOV = {len(vocabs[col]) + 1}')

## 4. Dataset and model

In [ ]:
class SparseCTRDataset(Dataset):
    def __init__(self, hf_dataset, feature_cols, vocabs, label_col='label_type'):
        self.data = hf_dataset
        self.feature_cols = feature_cols
        self.vocabs = vocabs
        self.label_col = label_col

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        x = []
        for col in self.feature_cols:
            raw = row[col]
            val = int(raw) if raw is not None else None
            x.append(self.vocabs[col].get(val, 0))
        y = to_binary_label(row[self.label_col])
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.float32)

BATCH_SIZE = 64
train_loader = DataLoader(SparseCTRDataset(train_data, feature_cols, vocabs), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SparseCTRDataset(val_data, feature_cols, vocabs), batch_size=256, shuffle=False)

xb, yb = next(iter(train_loader))
print('x batch shape:', xb.shape)
print('y batch shape:', yb.shape)
print('label min/max:', yb.min().item(), yb.max().item())

In [ ]:
class MultiFieldMLP(nn.Module):
    def __init__(self, vocab_sizes, embed_dim=16, hidden_dims=(64, 32), dropout=0.25):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(v, embed_dim) for v in vocab_sizes])
        input_dim = len(vocab_sizes) * embed_dim
        layers = []
        for h in hidden_dims:
            layers.append(nn.Linear(input_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        # x: [batch, num_fields]
        embs = [emb(x[:, i]) for i, emb in enumerate(self.embeddings)]
        z = torch.cat(embs, dim=1)
        logits = self.mlp(z).squeeze(1)
        return logits  # raw logits only; do NOT sigmoid here

vocab_sizes = [len(vocabs[c]) + 1 for c in feature_cols]
model = MultiFieldMLP(vocab_sizes=vocab_sizes, embed_dim=16).to(device)
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## 5. Stable training loop

Expected behavior: training loss should stay positive and usually decrease slowly. Validation AUC may bounce around because this is only a 1K sample.

In [ ]:
def evaluate(model, loader):
    model.eval()
    ys, probs = [], []
    total_loss = 0.0
    loss_fn = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            p = torch.sigmoid(logits)
            total_loss += loss.item() * x.size(0)
            probs.extend(p.cpu().numpy().tolist())
            ys.extend(y.cpu().numpy().tolist())

    val_loss = total_loss / len(loader.dataset)
    if len(set(ys)) < 2:
        auc = float('nan')
    else:
        auc = roc_auc_score(ys, probs)
    return val_loss, auc

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

EPOCHS = 15
best_auc = -1
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for x, y in tqdm(train_loader, desc=f'Epoch {epoch:02d}'):
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = loss_fn(logits, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * x.size(0)

    train_loss = total_loss / len(train_loader.dataset)
    val_loss, val_auc = evaluate(model, val_loader)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_auc': val_auc})

    if not math.isnan(val_auc) and val_auc > best_auc:
        best_auc = val_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch:02d} | train loss = {train_loss:.4f} | val loss = {val_loss:.4f} | val AUC = {val_auc:.4f}')

print('
Best validation AUC:', best_auc)
if best_state is not None:
    model.load_state_dict(best_state)

In [ ]:
# Show results table for the midterm report
results_df = pd.DataFrame(history)
results_df

In [ ]:
# Optional plot for your report
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(results_df['epoch'], results_df['val_auc'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('Validation AUC')
plt.title('Baseline MLP Validation AUC on 1K Sample')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Save outputs for report

In [ ]:
# Save the training history and model checkpoint in Colab runtime.
results_df.to_csv('baseline_training_history.csv', index=False)
torch.save({
    'model_state_dict': model.state_dict(),
    'feature_cols': feature_cols,
    'vocabs': vocabs,
    'best_auc': best_auc,
}, 'baseline_mlp_fixed.pt')
print('Saved baseline_training_history.csv and baseline_mlp_fixed.pt')

## 7. What to report in the midterm

Use the best validation AUC from this notebook as your local preliminary result.

Suggested wording:

> We implemented a corrected MLP baseline using scalar user/item sparse features from the Hugging Face 1K sample. The initial unstable run revealed a loss/label handling issue, which was fixed by using `BCEWithLogitsLoss`, raw logits, binary label conversion, gradient clipping, and regularization. After the fix, the training loss remained numerically stable and validation AUC was used as the local sanity-check metric.

Next model upgrade:

1. Add list sparse features.
2. Add sequence fields with truncation.
3. Replace MLP head with the UFI-Former unified token Transformer.
4. Submit predictions to official evaluation and screenshot leaderboard result.